# Lab 12 — Lifespan Events

**Difficulty: Intermediate | ~40 min | Requires Lab 4**

### Step 1: Install Dependencies

This cell installs every pinned dependency the lab needs. Run this first so all later cells have what they require.

In [1]:
!pip install fastapi==0.112.2 pydantic==2.8.2 httpx==0.28.1


[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


### Step 2: Imports

We import `FastAPI` for the application, `Depends` and `Request` for dependency injection, `asynccontextmanager` to build the lifespan context manager, and `time` to simulate an expensive load.

In [5]:
from fastapi import FastAPI, Depends, Request, HTTPException
from contextlib import asynccontextmanager
import time

### Step 3: The Expensive Load Function

This function simulates what a real application would do at startup — load an embedding model, build a retrieval index, or open a database connection pool. The `time.sleep(5)` stands in for real load cost. Both the naive and lifespan paths call this same function, so the only variable being compared is **when** the cost is paid.

In [6]:
def load_index():
    time.sleep(5)
    return {"doc1": "FastAPI is a modern Python web framework",
            "doc2": "Lifespan events run at startup and shutdown",
            "doc3": "app.state stores shared resources across requests"}

### Step 4: The Naive Path (No Lifespan)

The naive dependency `get_index_naive` calls `load_index()` fresh on every single invocation. This is the flaw being demonstrated — every request pays the full 5-second cost again, even though the result is always the same dict. The endpoint does a trivial lookup and returns the result.

In [7]:
def get_index_naive():
    return load_index()

app_naive = FastAPI()

@app_naive.get("/search-naive")
def search_naive(q: str, index=Depends(get_index_naive)):
    if q not in index:
        raise HTTPException(status_code=404, detail="not found")
    return {"query": q, "result": index[q]}

### Step 5: The Lifespan Function

The lifespan function uses `@asynccontextmanager` to turn a generator into an async context manager. Code before `yield` runs at startup — this is where the expensive load happens exactly once. Code after `yield` runs at shutdown — here, we clean up by setting the index to `None`. The `app` parameter gives access to `app.state`, which is where the loaded index is stored so every request can reach it.

In [8]:
@asynccontextmanager
async def lifespan(app: FastAPI):
    app.state.index = load_index()
    yield
    app.state.index = None

### Step 6: The Lifespan-Backed App and Dependency

We create a new FastAPI app with the `lifespan` parameter — this is what tells FastAPI to run the lifespan function at startup and shutdown. The `get_index` dependency does **no loading at all**: it simply reads `request.app.state.index`, which was already populated by the lifespan function before any request was accepted.

In [9]:
app = FastAPI(lifespan=lifespan)

def get_index(request: Request):
    return request.app.state.index

### Step 7: The Lifespan-Backed Search Endpoint

This endpoint's logic is identical to `search_naive` — the only difference is that its dependency reads from `app.state` instead of calling `load_index()` on every request. The lookup is trivial on purpose: the lab is measuring when the load runs, not what the endpoint does with the result.

In [10]:
@app.get("/search")
def search(q: str, index=Depends(get_index)):
    if q not in index:
        raise HTTPException(status_code=404, detail="not found")
    return {"query": q, "result": index[q]}

### Proof 1 — Naive Cost, Repeated

We instantiate `TestClient(app_naive)`. We send three sequential requests, timing each one. All three should take roughly 5 seconds, proving that `load_index()` reruns on every single call.

In [11]:
from fastapi.testclient import TestClient

client_naive = TestClient(app_naive)

for i, query in enumerate(["doc1", "doc2", "doc3"], 1):
    start = time.time()
    res = client_naive.get(f"/search-naive?q={query}")
    elapsed = time.time() - start
    print(f"Request {i}: status={res.status_code}, time={elapsed:.2f}s, "
          f"result={res.json()['result'][:40]}...")

Request 1: status=200, time=5.02s, result=FastAPI is a modern Python web framework...
Request 2: status=200, time=5.01s, result=Lifespan events run at startup and shutd...
Request 3: status=200, time=5.01s, result=app.state stores shared resources across...


### Proof 2 — Lifespan Cost, Paid Once

We use `with TestClient(app) as client:` — entering the `with` block is what triggers the lifespan startup, including the 5-second `load_index()` call. Exiting the block triggers shutdown. Inside the block, we send three sequential requests, each of which reads from `app.state.index` instantly. All three should be near-instant.

In [12]:
with TestClient(app) as client:
    for i, query in enumerate(["doc1", "doc2", "doc3"], 1):
        start = time.time()
        res = client.get(f"/search?q={query}")
        elapsed = time.time() - start
        print(f"Request {i}: status={res.status_code}, time={elapsed:.4f}s, "
              f"result={res.json()['result'][:40]}...")

Request 1: status=200, time=0.0023s, result=FastAPI is a modern Python web framework...
Request 2: status=200, time=0.0022s, result=Lifespan events run at startup and shutd...
Request 3: status=200, time=0.0011s, result=app.state stores shared resources across...


### Proof 3 — Loaded Up Front, Not Lazily

Still inside the same `with` block, we check that `app.state.index` is already populated before any request is sent — the cost was paid once, at startup, not on whichever request happened to arrive first.

In [13]:
with TestClient(app) as client:
    print(f"Index populated before first request? {client.app.state.index is not None}")
    print(f"Index keys: {list(client.app.state.index.keys())}")
    res = client.get("/search?q=doc1")
    print(f"Request after check: status={res.status_code}, "
          f"result={res.json()['result'][:40]}...")

Index populated before first request? True
Index keys: ['doc1', 'doc2', 'doc3']
Request after check: status=200, result=FastAPI is a modern Python web framework...


### Proof 4 — Cleanup Genuinely Runs

After the `with` block has closed, we assert that `app.state.index` is now `None` — proving the code written after `yield` in the lifespan function actually executed, not just that it was syntactically present.

In [14]:
assert app.state.index is None, "Expected app.state.index to be None after lifespan shutdown"
print("Cleanup verified: app.state.index is None after with block closed")

Cleanup verified: app.state.index is None after with block closed
